In [1]:
import pandas as pd
import wandb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os
warnings.filterwarnings('ignore')

def setup_publication_style():
    """Configure matplotlib for publication-quality figures following ICML standards (half-page width)."""
    # Set up colorblind-friendly colors (Tol palette)
    colors = {
        'primary': '#1f77b4',      # Blue
        'secondary': '#ff7f0e',    # Orange  
        'tertiary': '#2ca02c',     # Green
        'quaternary': '#d62728',   # Red
        'quinary': '#9467bd',      # Purple
        'senary': '#8c564b',       # Brown
        'grid': '#808080',         # Gray
    }
    
    # Configure matplotlib for publication (ICML standards - half page width)
    plt.rcParams.update({
        'font.size': 10,           # Smaller base font for half-page
        'axes.labelsize': 11,      # Axis label size
        'legend.fontsize': 9,      # Smaller legend font
        'xtick.labelsize': 9,      # X tick label size
        'ytick.labelsize': 9,      # Y tick label size
        'lines.linewidth': 1.8,    # Line width
        'lines.markersize': 4,     # Smaller markers for half-page
        'font.family': 'serif',    # Font family
        'text.usetex': False,      # LaTeX rendering
        'axes.grid': True,         # Grid on by default
        'axes.axisbelow': True,    # Grid below data
        'grid.alpha': 0.3,         # Grid transparency
        'grid.linewidth': 0.6,     # Thinner grid for half-page
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'axes.spines.top': False,    # Remove top spine
        'axes.spines.right': False,  # Remove right spine
        'axes.spines.left': True,    # Keep left spine
        'axes.spines.bottom': True,  # Keep bottom spine
    })
    
    return colors

class ComprehensiveGradientAnalysis:
    def __init__(self, run_id, num_timesteps=10):
        """
        Comprehensive analysis with LR tracking, tables, and heatmaps
        """
        self.api = wandb.Api()
        self.run_id = run_id
        self.num_timesteps = num_timesteps
        
        # Load run and history
        self.run = self.api.run(run_id)
        self.history = self.run.history(samples=100000)
        
        print(f"📊 Loaded run: {self.run.name}")
        print(f"📈 History shape: {self.history.shape}")
        
        # Define layer groups for gradient analysis
        self.layer_groups = {
            'embedding': ['embedding', 'embed'],
            'n_layers': ['n_layers', 'n_layer'],
            'k_layers': ['k_layers', 'k_layer'], 
            'l_layers': ['l_layers', 'l_layer'],
            'p_layers': ['p_layers', 'p_layer'],
            'm_layers': ['m_layers', 'm_layer']
        }
        
        # Extract rescale factors from config
        self.rescale_factors = self._extract_rescale_factors()
        
    def _extract_rescale_factors(self):
        """Extract gradient rescaling factors from wandb config"""
        config = self.run.config
        rescale_factors = {}
        
        if 'model' in config and isinstance(config['model'], dict):
            model_config = config['model']
            if 'params' in model_config and 'gradient_rescaling' in model_config['params']:
                rescale_factors = model_config['params']['gradient_rescaling']
            elif 'gradient_rescaling' in model_config:
                rescale_factors = model_config['gradient_rescaling']
        
        print(f"🔧 Found rescale factors: {rescale_factors}")
        return rescale_factors
    
    def _extract_packed_histogram_stats(self, histogram_data):
        """Extract statistics from WandB's packedBins histogram format"""
        if not isinstance(histogram_data, dict) or histogram_data.get('_type') != 'histogram':
            return None
        
        try:
            # Extract packed bins info
            packed_bins = histogram_data.get('packedBins', {})
            values = np.array(histogram_data.get('values', []))
            
            if len(values) == 0:
                return None
            
            # Reconstruct bins from packed format
            bin_count = packed_bins.get('count', len(values))
            bin_min = packed_bins.get('min', 0.0)
            bin_size = packed_bins.get('size', 1.0)
            
            # Create bin edges
            bin_edges = np.linspace(bin_min, bin_min + bin_size * bin_count, bin_count + 1)
            bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
            
            # Ensure we have the right number of bins
            if len(bin_centers) != len(values):
                bin_centers = bin_centers[:len(values)]
            
            # Calculate statistics
            total_count = np.sum(values)
            if total_count == 0:
                return None
            
            # Weighted statistics
            mean = np.sum(bin_centers * values) / total_count
            variance = np.sum(((bin_centers - mean) ** 2) * values) / total_count
            std = np.sqrt(variance) if variance >= 0 else 0.0
            
            return {
                'mean': float(mean),
                'std': float(std),
                'abs_mean': float(abs(mean)),
                'count': int(total_count)
            }
        except Exception as e:
            return None
    
    def extract_comprehensive_data(self):
        """Extract all data including gradients, metrics, and learning rates"""
        gradient_cols = [col for col in self.history.columns if 'gradient' in col.lower()]
        
        # Find timesteps with complete data (skip step 0)
        valid_indices = []
        for idx in range(1, len(self.history)):
            # Check for gradient data
            has_gradient_data = any(not pd.isna(self.history.iloc[idx][col]) for col in gradient_cols[:3])
            # Check for metric data
            has_metric_data = not pd.isna(self.history.iloc[idx].get('val/metric/accuracy', np.nan))
            
            if has_gradient_data and has_metric_data:
                valid_indices.append(idx)
        
        if len(valid_indices) == 0:
            print("⚠️ No complete data found")
            return None
        
        # Select n timesteps linearly across the timeline
        if len(valid_indices) >= self.num_timesteps:
            selected_indices = np.linspace(0, len(valid_indices)-1, self.num_timesteps, dtype=int)
            timesteps = [valid_indices[i] for i in selected_indices]
        else:
            timesteps = valid_indices
        
        print(f"📊 Selected {len(timesteps)} timesteps for comprehensive analysis")
        
        # Extract data for each timestep
        comprehensive_data = []
        gradient_matrix = []  # For heatmap
        gradient_matrix_headers = []
        
        for timestep_idx, actual_idx in enumerate(timesteps):
            step = self.history.iloc[actual_idx].get('step', actual_idx)
            row_data = self.history.iloc[actual_idx]
            
            # Basic info
            data_point = {
                'timestep': timestep_idx + 1,
                'step': int(step),
                'actual_idx': actual_idx
            }
            
            # Learning rates
            lr_cols = [col for col in self.history.columns if 'hyperparameter/lr_' in col]
            base_lr = row_data.get('hyperparameter/lr_base_scheduler', np.nan)
            data_point['base_lr'] = base_lr
            
            # Extract effective LRs for each layer group
            for group_name in self.layer_groups.keys():
                effective_lr_col = f'hyperparameter/lr_effective_{group_name}'
                if effective_lr_col in self.history.columns:
                    data_point[f'lr_{group_name}'] = row_data.get(effective_lr_col, np.nan)
            
            # Training metrics
            data_point['train_accuracy'] = row_data.get('train/metric/accuracy', np.nan)
            data_point['val_accuracy'] = row_data.get('val/metric/accuracy', np.nan)
            data_point['train_bleu'] = row_data.get('train/metric/bleu', np.nan)
            data_point['val_bleu'] = row_data.get('val/metric/bleu', np.nan)
            data_point['train_loss'] = row_data.get('train/loss/main', np.nan)
            data_point['val_loss'] = row_data.get('val/loss/main', np.nan)
            
            # Gradient data for each layer group
            gradient_row = []
            for group_name, group_patterns in self.layer_groups.items():
                # Find gradient columns for this layer group
                group_cols = []
                for col in gradient_cols:
                    if any(pattern in col for pattern in group_patterns):
                        group_cols.append(col)
                
                # Aggregate statistics across all parameters in this group
                group_stats = []
                for col in group_cols:
                    hist_data = row_data[col]
                    if pd.isna(hist_data):
                        continue
                    
                    stats = self._extract_packed_histogram_stats(hist_data)
                    if stats is not None:
                        group_stats.append(stats)
                
                if group_stats:
                    grad_mean = np.mean([s['mean'] for s in group_stats])
                    grad_abs_mean = np.mean([s['abs_mean'] for s in group_stats])
                    grad_std = np.mean([s['std'] for s in group_stats])
                    
                    data_point[f'grad_mean_{group_name}'] = grad_mean
                    data_point[f'grad_abs_mean_{group_name}'] = grad_abs_mean
                    data_point[f'grad_std_{group_name}'] = grad_std
                    
                    gradient_row.append(grad_abs_mean)
                else:
                    data_point[f'grad_mean_{group_name}'] = np.nan
                    data_point[f'grad_abs_mean_{group_name}'] = np.nan
                    data_point[f'grad_std_{group_name}'] = np.nan
                    gradient_row.append(np.nan)
            
            comprehensive_data.append(data_point)
            gradient_matrix.append(gradient_row)
            
            if timestep_idx == 0:  # Set headers once
                gradient_matrix_headers = list(self.layer_groups.keys())
        
        # Convert to DataFrames
        df = pd.DataFrame(comprehensive_data)
        gradient_heatmap_data = pd.DataFrame(gradient_matrix, 
                                           columns=gradient_matrix_headers,
                                           index=[f"Step {row['step']}" for row in comprehensive_data])
        
        return df, gradient_heatmap_data
    
    def create_separate_plots(self, df, gradient_heatmap_data, output_dir="."):
        """Create separate publication-quality plots for ICML (half-page width)"""
        if df is None or df.empty:
            print("⚠️ No data to plot")
            return None
        
        colors = setup_publication_style()
        
        # Define colors for layer groups
        layer_colors = plt.cm.Set3(np.linspace(0, 1, len(self.layer_groups)))
        group_colors = dict(zip(self.layer_groups.keys(), layer_colors))
        
        # Plot 1: Learning Rate Evolution
        fig1, ax = plt.subplots(figsize=(4, 3))
        
        if 'base_lr' in df.columns:
            valid_base_lr = df[df['base_lr'].notna()]
            if not valid_base_lr.empty:
                ax.plot(valid_base_lr['step'], valid_base_lr['base_lr'], 
                       marker='o', linewidth=2, color='black', 
                       markersize=4, label='Base LR')
        
        for i, group_name in enumerate(self.layer_groups.keys()):
            lr_col = f'lr_{group_name}'
            if lr_col in df.columns:
                valid_data = df[df[lr_col].notna()]
                if not valid_data.empty:
                    ax.plot(valid_data['step'], valid_data[lr_col], 
                           marker='s', linewidth=1.5, color=layer_colors[i], 
                           markersize=3, alpha=0.8, label=f'{group_name}')
        
        ax.set_xlabel('Training Step')
        ax.set_ylabel('Learning Rate')
        ax.set_yscale('log')
        ax.legend(fontsize=8, ncol=2)
        plt.tight_layout()
        
        # Save plot
        pdf_path = os.path.join(output_dir, "learning_rate_evolution.pdf")
        fig1.savefig(pdf_path, format='pdf', dpi=300, bbox_inches='tight')
        png_path = os.path.join(output_dir, "learning_rate_evolution.png")
        fig1.savefig(png_path, format='png', dpi=300, bbox_inches='tight')
        plt.close(fig1)
        
        # Plot 2: Accuracy Evolution
        fig2, ax = plt.subplots(figsize=(4, 3))
        
        if 'train_accuracy' in df.columns:
            valid_data = df[df['train_accuracy'].notna()]
            if not valid_data.empty:
                ax.plot(valid_data['step'], valid_data['train_accuracy'], 
                       marker='o', linewidth=2, color=colors['primary'], 
                       markersize=4, label='Train')
        
        if 'val_accuracy' in df.columns:
            valid_data = df[df['val_accuracy'].notna()]
            if not valid_data.empty:
                ax.plot(valid_data['step'], valid_data['val_accuracy'], 
                       marker='s', linewidth=2, color=colors['secondary'], 
                       markersize=4, label='Validation')
        
        ax.set_xlabel('Training Step')
        ax.set_ylabel('Accuracy')
        ax.legend()
        plt.tight_layout()
        
        # Save plot
        pdf_path = os.path.join(output_dir, "accuracy_evolution.pdf")
        fig2.savefig(pdf_path, format='pdf', dpi=300, bbox_inches='tight')
        png_path = os.path.join(output_dir, "accuracy_evolution.png")
        fig2.savefig(png_path, format='png', dpi=300, bbox_inches='tight')
        plt.close(fig2)
        
        # Plot 3: BLEU Evolution
        fig3, ax = plt.subplots(figsize=(4, 3))
        
        if 'train_bleu' in df.columns:
            valid_data = df[df['train_bleu'].notna()]
            if not valid_data.empty:
                ax.plot(valid_data['step'], valid_data['train_bleu'], 
                       marker='o', linewidth=2, color=colors['tertiary'], 
                       markersize=4, label='Train')
        
        if 'val_bleu' in df.columns:
            valid_data = df[df['val_bleu'].notna()]
            if not valid_data.empty:
                ax.plot(valid_data['step'], valid_data['val_bleu'], 
                       marker='s', linewidth=2, color=colors['quaternary'], 
                       markersize=4, label='Validation')
        
        ax.set_xlabel('Training Step')
        ax.set_ylabel('BLEU Score')
        ax.legend()
        plt.tight_layout()
        
        # Save plot
        pdf_path = os.path.join(output_dir, "bleu_evolution.pdf")
        fig3.savefig(pdf_path, format='pdf', dpi=300, bbox_inches='tight')
        png_path = os.path.join(output_dir, "bleu_evolution.png")
        fig3.savefig(png_path, format='png', dpi=300, bbox_inches='tight')
        plt.close(fig3)
        
        # Plot 4: Loss Evolution
        fig4, ax = plt.subplots(figsize=(4, 3))
        
        if 'train_loss' in df.columns:
            valid_data = df[df['train_loss'].notna()]
            if not valid_data.empty:
                ax.plot(valid_data['step'], valid_data['train_loss'], 
                       marker='o', linewidth=2, color=colors['quinary'], 
                       markersize=4, label='Train')
        
        if 'val_loss' in df.columns:
            valid_data = df[df['val_loss'].notna()]
            if not valid_data.empty:
                ax.plot(valid_data['step'], valid_data['val_loss'], 
                       marker='s', linewidth=2, color=colors['senary'], 
                       markersize=4, label='Validation')
        
        ax.set_xlabel('Training Step')
        ax.set_ylabel('Loss')
        ax.set_yscale('log')
        ax.legend()
        plt.tight_layout()
        
        # Save plot
        pdf_path = os.path.join(output_dir, "loss_evolution.pdf")
        fig4.savefig(pdf_path, format='pdf', dpi=300, bbox_inches='tight')
        png_path = os.path.join(output_dir, "loss_evolution.png")
        fig4.savefig(png_path, format='png', dpi=300, bbox_inches='tight')
        plt.close(fig4)
        
        # Plot 5: Gradient Absolute Mean Evolution
        fig5, ax = plt.subplots(figsize=(4, 3))
        
        for i, group_name in enumerate(self.layer_groups.keys()):
            grad_col = f'grad_abs_mean_{group_name}'
            if grad_col in df.columns:
                valid_data = df[df[grad_col].notna()]
                if not valid_data.empty:
                    ax.plot(valid_data['step'], valid_data[grad_col], 
                           marker='o', linewidth=1.8, color=layer_colors[i], 
                           markersize=3, label=group_name)
        
        ax.set_xlabel('Training Step')
        ax.set_ylabel('Gradient Absolute Mean')
        ax.set_yscale('log')
        ax.legend(fontsize=8, ncol=2)
        plt.tight_layout()
        
        # Save plot
        pdf_path = os.path.join(output_dir, "gradient_abs_mean_evolution.pdf")
        fig5.savefig(pdf_path, format='pdf', dpi=300, bbox_inches='tight')
        png_path = os.path.join(output_dir, "gradient_abs_mean_evolution.png")
        fig5.savefig(png_path, format='png', dpi=300, bbox_inches='tight')
        plt.close(fig5)
        
        # Plot 6: Gradient Mean Evolution
        fig6, ax = plt.subplots(figsize=(4, 3))
        
        for i, group_name in enumerate(self.layer_groups.keys()):
            grad_col = f'grad_mean_{group_name}'
            if grad_col in df.columns:
                valid_data = df[df[grad_col].notna()]
                if not valid_data.empty:
                    ax.plot(valid_data['step'], valid_data[grad_col], 
                           marker='s', linewidth=1.8, color=layer_colors[i], 
                           markersize=3, label=group_name)
        
        ax.set_xlabel('Training Step')
        ax.set_ylabel('Gradient Mean')
        ax.axhline(y=0, color='black', linestyle='--', alpha=0.5, linewidth=1)
        ax.legend(fontsize=8, ncol=2)
        plt.tight_layout()
        
        # Save plot
        pdf_path = os.path.join(output_dir, "gradient_mean_evolution.pdf")
        fig6.savefig(pdf_path, format='pdf', dpi=300, bbox_inches='tight')
        png_path = os.path.join(output_dir, "gradient_mean_evolution.png")
        fig6.savefig(png_path, format='png', dpi=300, bbox_inches='tight')
        plt.close(fig6)
        
        # Plot 7: Gradient Heatmap
        fig7, ax = plt.subplots(figsize=(4, 3))
        
        if gradient_heatmap_data is not None and not gradient_heatmap_data.empty:
            # Create heatmap data with proper handling of NaN values
            heatmap_data = gradient_heatmap_data.fillna(1e-12)
            
            # Use log scale for better visualization
            log_data = np.log10(heatmap_data + 1e-12)
            
            im = ax.imshow(log_data.T, cmap='viridis', aspect='auto', 
                          interpolation='nearest')
            
            # Set ticks and labels
            ax.set_xticks(range(0, len(gradient_heatmap_data.index), 2))  # Every 2nd tick
            ax.set_xticklabels([gradient_heatmap_data.index[i] for i in range(0, len(gradient_heatmap_data.index), 2)], 
                              rotation=45, fontsize=8)
            ax.set_yticks(range(len(gradient_heatmap_data.columns)))
            ax.set_yticklabels(gradient_heatmap_data.columns, fontsize=8)
            
            ax.set_xlabel('Training Steps')
            ax.set_ylabel('Layer Groups')
            
            # Add colorbar
            cbar = plt.colorbar(im, ax=ax, shrink=0.8)
            cbar.set_label('log₁₀(Gradient Abs Mean)', fontsize=9)
            cbar.ax.tick_params(labelsize=8)
            
        plt.tight_layout()
        
        # Save plot
        pdf_path = os.path.join(output_dir, "gradient_heatmap.pdf")
        fig7.savefig(pdf_path, format='pdf', dpi=300, bbox_inches='tight')
        png_path = os.path.join(output_dir, "gradient_heatmap.png")
        fig7.savefig(png_path, format='png', dpi=300, bbox_inches='tight')
        plt.close(fig7)
        
        print(f"💾 All plots saved to: {output_dir}/")
        return True

def log_with_timestamp(message):
    """Log a message with a timestamp"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] {message}")

def create_latex_figure_templates():
    """Generate LaTeX figure templates for ICML paper"""
    templates = {
        'single_plot': '''
% LaTeX figure template for single plot (half-page width)
\\begin{figure}[htbp]
    \\centering
    \\includegraphics[width=0.48\\textwidth]{filename.pdf}
    \\caption{Your caption here}
    \\label{fig:your_label}
\\end{figure}
''',
        'two_plots': '''
% LaTeX figure template for two plots side by side
\\begin{figure}[htbp]
    \\centering
    \\begin{subfigure}[b]{0.48\\textwidth}
        \\centering
        \\includegraphics[width=\\textwidth]{plot1.pdf}
        \\caption{Caption for plot 1}
        \\label{fig:plot1}
    \\end{subfigure}
    \\hfill
    \\begin{subfigure}[b]{0.48\\textwidth}
        \\centering
        \\includegraphics[width=\\textwidth]{plot2.pdf}
        \\caption{Caption for plot 2}
        \\label{fig:plot2}
    \\end{subfigure}
    \\caption{Overall caption for both plots}
    \\label{fig:combined}
\\end{figure}
''',
        'comprehensive': '''
% LaTeX figure template for comprehensive gradient analysis
\\begin{figure*}[htbp]
    \\centering
    \\begin{subfigure}[b]{0.32\\textwidth}
        \\centering
        \\includegraphics[width=\\textwidth]{learning_rate_evolution.pdf}
        \\caption{Learning rate evolution}
        \\label{fig:lr_evolution}
    \\end{subfigure}
    \\hfill
    \\begin{subfigure}[b]{0.32\\textwidth}
        \\centering
        \\includegraphics[width=\\textwidth]{accuracy_evolution.pdf}
        \\caption{Accuracy evolution}
        \\label{fig:acc_evolution}
    \\end{subfigure}
    \\hfill
    \\begin{subfigure}[b]{0.32\\textwidth}
        \\centering
        \\includegraphics[width=\\textwidth]{gradient_abs_mean_evolution.pdf}
        \\caption{Gradient magnitude evolution}
        \\label{fig:grad_evolution}
    \\end{subfigure}
    
    \\vspace{0.5cm}
    
    \\begin{subfigure}[b]{0.32\\textwidth}
        \\centering
        \\includegraphics[width=\\textwidth]{bleu_evolution.pdf}
        \\caption{BLEU score evolution}
        \\label{fig:bleu_evolution}
    \\end{subfigure}
    \\hfill
    \\begin{subfigure}[b]{0.32\\textwidth}
        \\centering
        \\includegraphics[width=\\textwidth]{loss_evolution.pdf}
        \\caption{Loss evolution}
        \\label{fig:loss_evolution}
    \\end{subfigure}
    \\hfill
    \\begin{subfigure}[b]{0.32\\textwidth}
        \\centering
        \\includegraphics[width=\\textwidth]{gradient_heatmap.pdf}
        \\caption{Gradient heatmap}
        \\label{fig:grad_heatmap}
    \\end{subfigure}
    
    \\caption{Comprehensive training analysis showing learning rate schedules, performance metrics, and gradient behavior across different layer groups during RDDLGN training. All plots show evolution over training steps.}
    \\label{fig:comprehensive_analysis}
\\end{figure*}
'''
    }
    
    print("\n" + "="*80)
    print("LATEX FIGURE TEMPLATES FOR ICML PAPER:")
    print("="*80)
    for template_name, template_code in templates.items():
        print(f"\n{template_name.upper()} TEMPLATE:")
        print("-" * 50)
        print(template_code)
    print("="*80)

def main():
    """Main function to orchestrate the analysis"""
    log_with_timestamp("=== COMPREHENSIVE GRADIENT ANALYSIS STARTED ===")
    
    # Configuration
    run_id = "sbuehrer-eth-z-rich/RDDLGN/bfn0q6b0"
    
    # Create output directory
    output_dir = "gradient_analysis_icml"
    os.makedirs(output_dir, exist_ok=True)
    
    print("🚀 Initializing Comprehensive Gradient Analysis...")
    analyzer = ComprehensiveGradientAnalysis(run_id, num_timesteps=10)
    
    print("📊 Extracting comprehensive data...")
    result = analyzer.extract_comprehensive_data()
    
    if result is None:
        print("❌ No data extracted.")
        return
    
    df, gradient_heatmap_data = result
    print(f"✅ Extracted data for {len(df)} timesteps")
    
    # Create separate publication-quality plots
    log_with_timestamp("Creating separate ICML-style plots...")
    success = analyzer.create_separate_plots(df, gradient_heatmap_data, output_dir)
    
    if success:
        log_with_timestamp(f"All plots saved to: {output_dir}/")
        
        # Save data to CSV for further analysis
        df.to_csv(os.path.join(output_dir, "training_analysis_data.csv"), index=False)
        gradient_heatmap_data.to_csv(os.path.join(output_dir, "gradient_heatmap_data.csv"))
        log_with_timestamp("Data exported to CSV files")
        
        # Generate LaTeX templates
        create_latex_figure_templates()
        
        log_with_timestamp("=== ANALYSIS COMPLETED ===")
    else:
        log_with_timestamp("⚠️ No plots generated - insufficient data")

if __name__ == "__main__":
    main()

[2025-07-09 07:57:32] === COMPREHENSIVE GRADIENT ANALYSIS STARTED ===
🚀 Initializing Comprehensive Gradient Analysis...
📊 Loaded run: unsynced_UnsyncedRecurrentDifflogic_AdamW_lr5e-02_0704_1347_grad_upd_collapse
📈 History shape: (518, 218)
🔧 Found rescale factors: {'k_layers': 1, 'l_layers': 1, 'm_layers': 1, 'n_layers': 1, 'p_layers': 1, 'embedding': 1}
📊 Extracting comprehensive data...
📊 Selected 10 timesteps for comprehensive analysis
✅ Extracted data for 10 timesteps
[2025-07-09 07:57:34] Creating separate ICML-style plots...
💾 All plots saved to: gradient_analysis_icml/
[2025-07-09 07:57:41] All plots saved to: gradient_analysis_icml/
[2025-07-09 07:57:41] Data exported to CSV files

LATEX FIGURE TEMPLATES FOR ICML PAPER:

SINGLE_PLOT TEMPLATE:
--------------------------------------------------

% LaTeX figure template for single plot (half-page width)
\begin{figure}[htbp]
    \centering
    \includegraphics[width=0.48\textwidth]{filename.pdf}
    \caption{Your caption here}
    